# 单例模式 & 魔术方法
在面向对象编程中，除了基础的类和对象，还有两个实用高级特性——**单例模式** 和 **魔术方法**。

- **单例模式**：就像给类盖了“唯一印章”，确保在整个程序中只能创建一个实例  
  （例如电脑的回收站，全系统只有一个）。  
  👉 好处：避免资源重复占用，保证数据一致性。

- **魔术方法**（以双下划线包裹的方法）：相当于给对象增加“特效按钮”，让对象具备特殊行为  
  （例如通过 `__str__` 自定义打印输出内容）。

虽然它们被称为“高级特性”，但核心目标其实很简单：  
👉 **让面向对象编程更贴近实际需求**

掌握这两个特性，可以让你的代码：
- 跳出基础语法的限制  
- 更优雅地处理复杂场景  
- 写出更清晰、高效的面向对象程序

---

# 1.  对象实例化的秘密（`__new__` vs `__init__`）  
对象实例化时，真正最先执行的是`__new__`方法。

## 1.1 通俗理解：造房子的比喻

- `__new__`：负责“打地基、造房子框架”  
  👉 在内存中为对象分配空间，并返回对象的引用（相当于房子的“门牌号”）

- `__init__`：负责“装修房子、放家具”  
  👉 接收 `__new__` 返回的对象引用，为对象初始化属性（例如 `self.name`）


### ✅ 简单总结

- `__new__`：创建对象  
- `__init__`：初始化对象  
- 二者缺一不可

## 1.2 实例化的完整流程
1. 调用`类名()`时,Python先执行`__new__`方法，分配内存空间，返回对象引用；
2. Python 把对象引用作为`self`参数，传给`__init__`方法；
3. `__init__`方法给对象设置属性，完成初始化。

In [2]:
class House:
    def __new__(cls,*args,**kwargs):
        # cls代表类本身，必须作为一个参数
        print("__new__: 分配内存，造房子框架")
        # 调用父类的__new__方法，分配内存，返回对象引用
        obj = object.__new__(cls)
        return obj                  #重写new方法必须返回对象引用，否则会导致__init__方法无法被调用

    def __init__(self,address,area):
        print("__init__: 初始化对象，装修房子，设置地址和面积")
        self.address = address
        self.area = area

# 实例化对象（触发__new__ 和 __init__ 方法）
house1 = House("北京市朝阳区", 120)

# __new__方法返回的对象引用会被传递给__init__方法作为第一个参数（self），因此在__init__方法中可以使用self来访问和设置对象的属性。

__new__: 分配内存，造房子框架
__init__: 初始化对象，装修房子，设置地址和面积


## 1.3 核心区别对比

| 对比维度 | `__new__` 方法 | `__init__` 方法 |
|----------|--------------|----------------|
| 核心作用 | 创建对象、分配内存 | 初始化对象、设置属性 |
| 第一个参数 | `cls`（代表当前类） | `self`（代表当前对象） |
| 返回值 | **必须返回对象引用** | 无返回值（默认 `None`） |
| 执行时机 | 实例化时先执行 | 在 `__new__` 之后执行 |

---

### ✅ 一句话总结
- `__new__`：负责“生出来”  
- `__init__`：负责“养起来”

---

# 2.  单例模式：一个类只能创建一个实例
## 2.1 什么是单例模式?
单例模式是一种设计模式，核心是"让一个类在整个程序中，只能创建一个实例对象"-- 就像电脑的回收站，不管你双击多少次，都只会打开同一个窗口;再比如公司的打印机，所有人都用同一台，不会每人专属一台(浪费资源)。


## 2.2 核心优势
- **节省内存**：避免创建多个重复对象，减少资源浪费(比如数据库连接，创建一次供全程序使用)
- **统一管理**：全程序使用同一个对象，数据一致，避免冲突(比如游戏的场景管理器)


## 2.3 重点实现：重写`__new__`方法

### 单例模式：设计思路

1. **定义类属性**
   - 例如 `_instance`，初始值为 `None`  
   - 用于记录唯一的实例

2. **重写 `__new__` 方法**
   - 如果类属性是 `None`：
     - 调用父类 `__new__` 创建实例
     - 将实例保存到类属性中
   - 如果类属性不是 `None`：
     - 直接返回已存在的实例

### ✅ 效果
- 无论实例化多少次，都只会返回**同一个对象**

In [7]:
class Singleton:
    _instance = None

    # 重写__new__方法，实现单例模式
    def __new__(cls,*args,**kwargs):
        if cls._instance is None:
            cls._instance = object.__new__(cls)
        return cls._instance
    
    def __init__(self,name):
        # 每次实例化都会执行__init__方法，需避免重复初始化
        # hasattr函数检查对象是否已经有name属性，如果没有则进行初始化
        if not hasattr(self, 'name'):
            self.name = name



object1 = Singleton("o1")
print(object1.name)
object2 = Singleton("o2")
print(object1.name)

print(object2.name)

o1
o1
o1


#### 关键提醒
Python的模块是天然的单例模式--模块第一次导入时，会生成 `.pyc` 文件，后续导入不会重新执行模块代码，只会加载已生成的文件。

## 2.4.应用场景
1. **系统工具类**:如回收站、打印机、音乐播放器(同一时间只能播放一首歌曲)。
2. **资源密集型对象**:如数据库连接池、网络连接(创建成本高，复用更高效)。
3. **全局管理器**:如游戏场景管理器、配置管理器(需要统一管理数据/状态)。

---

# 3. 魔术方法 & 魔术属性

## 3.1. 什么是魔术方法 / 魔术属性？

Python 中，以 __xx__ 命名的方法 / 属性，叫做 “魔术方法 / 属性”，它们有特殊功能，能让对象拥有额外能力（比如打印对象时显示友好信息）。

## 3.2. 常用魔术属性
| 魔术属性   | 核心作用                         | 示例                          |
|------------|----------------------------------|-------------------------------|
| `__doc__`    | 查看类的描述信息（三引号定义）   | `print(类名.__doc__)`          |
| `__module__` | 查看对象所属的模块               | `print(对象.__module__)`        |
| `__class__`  | 查看对象所属的类                 | `print(对象.__class__)`         |
| `__dict__`   | 查看对象的属性和值（字典形式）   | `print(对象.__dict__)`          |

In [9]:
# 定义学生类
class Student:
    """学生类，包含姓名和年龄属性"""
    def __init__(self, name, age):
        self.name = name
        self.age = age 

# 创建学生对象
student1 = Student("Alice", 20)
print(student1.__doc__)  # 输出学生对象的文档字符串
print(student1.__module__)  # 输出学生对象的模块名称 (当前模块运行时为__main__，如果在其他模块中导入则为模块名称)
print(student1.__class__)  # 输出学生对象的类信息
print(student1.__dict__)  # 输出学生对象的属性字典

学生类，包含姓名和年龄属性
__main__
<class '__main__.Student'>
{'name': 'Alice', 'age': 20}


## 3.3 常用魔术方法
### 3.3.1 `__str__()` 自定义对象的打印信息
默认情况下，打印对象会显示内存地址(不友好) `__str__` 能自定义打印内容，必须返回字符串。

In [13]:
class Student:
    """学生类，包含姓名和年龄属性"""
    def __init__(self, name, age):
        self.name = name
        self.age = age
    
    def __str__(self):
        return f"学生姓名: {self.name}, 年龄: {self.age}"       #必须返回字符串，否则会导致TypeError异常
    
student1 = Student("Alice", 20)
print(student1)         # 打印对象自动调用__str__方法，输出学生信息

学生姓名: Alice, 年龄: 20


### 3.3.2 `__del__()` 对象被删除时触发
当对象被删除（手动 `del` 或程序结束），Python 会自动调用`__del__`，可用于释放资源（如关闭文件）。


### 3.3.3. `__call__()`：让对象能像函数一样调用

默认情况下，对象不能像函数那样加 () 调用，`__call__`能实现这个功能。

In [14]:
class Calculator:
    def __call__(self, a, b):
        # 对象调用时执行的逻辑
        return a + b

# 创建对象
calc = Calculator()

# 像函数一样调用对象
result = calc(3, 5)
print(result)  # 输出：8
print(callable(calc))  # 输出：True（判断是否可调用）

8
True


#### 避坑提示

- __str__ 方法必须返回字符串，否则会报错；
- __call__ 方法的参数可以自定义，和普通函数一样；
- __del__ 方法不用手动调用，Python 会自动处理，避免手动调用导致异常。

## 4. 核心总结

1. 实例化核心：

   - 流程：__new__（分配空间 → 返回引用） → __init__（初始化属性）；
   - 关键：__new__ 必须返回对象引用，否则 __init__ 不执行。

2. 单例模式：

   - 核心：一个类只能创建一个实例；
   - 常用实现：重写 __new__，用类属性记录唯一实例；
   - 优势：省内存、统一管理；
   - 应用场景：工具类、资源密集型对象、全局管理器。

3. 魔术方法 & 属性：

   - 魔术属性：__doc__（描述）、__dict__（属性字典）、__module__（所属模块）；
   - 魔术方法：__str__（自定义打印）、__del__（对象删除触发）、__call__（对象可调用）；
   - 特点：__xx__ 格式，自带特殊功能，无需手动调用。